# Step 3: BiLSTM
`BiLSTM` (pack → LSTM → unpack) is tested here (one notebook per stage), on hand-made `(B, T, 32)` inputs, so no vocab or embedding is involved: a failure here can only come from this stage.
The class itself lives in `08_encoder_step.py`.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from importlib import import_module
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import torch
from torch import nn


In [2]:
# the class lives in 08_encoder_step.py (single copy, shared with training); this notebook tests it
BiLSTM = import_module("siamese_model.08_encoder_step").BiLSTM

## Tests

In [3]:
torch.manual_seed(0)
lengths = torch.tensor([3, 1, 6])
x = torch.randn(3, 6, 32)   # pad positions are random garbage on purpose: packing must ignore them
bilstm = BiLSTM()
bilstm.eval()
with torch.no_grad():
    h = bilstm(x, lengths)
    h_single = bilstm(x[2:3], lengths[2:3])   # row 2 on its own

assert h.shape == (3, 6, 256)                        # (B, T, 2 * hidden): forward 128 + backward 128
assert torch.equal(h[1, 1:], torch.zeros(5, 256))    # unpacked pad positions are exactly 0 (row 1 has length 1)
assert torch.equal(h[0, 3:], torch.zeros(3, 256))
# row order survives enforce_sorted=False, and other rows don't leak in;
# allclose, not equal: batched vs single differ by ~3e-8 of float rounding
assert torch.allclose(h[2], h_single[0], atol=1e-6)

# garbage in the pad positions changes nothing (the LSTM never saw them)
x_other_pads = x.clone()
x_other_pads[1, 1:] = 100.0
with torch.no_grad():
    assert torch.allclose(bilstm(x_other_pads, lengths), h, atol=1e-6)
print("BiLSTM shapes / padding: all passed")

BiLSTM shapes / padding: all passed


In [4]:
# parameters: shapes by name
params = dict(bilstm.named_parameters())
for k, v in params.items():
    print(f"{k}: {tuple(v.shape)}")
for suffix in ["", "_reverse"]:
    assert params[f"lstm.weight_ih_l0{suffix}"].shape == (512, 32)    # 4 gates x 128 hidden, reading the 32-dim input
    assert params[f"lstm.weight_hh_l0{suffix}"].shape == (512, 128)   # 4 gates x 128 hidden, reading the previous hidden state
    assert params[f"lstm.bias_ih_l0{suffix}"].shape == (512,)
    assert params[f"lstm.bias_hh_l0{suffix}"].shape == (512,)

lstm.weight_ih_l0: (512, 32)
lstm.weight_hh_l0: (512, 128)
lstm.bias_ih_l0: (512,)
lstm.bias_hh_l0: (512,)
lstm.weight_ih_l0_reverse: (512, 32)
lstm.weight_hh_l0_reverse: (512, 128)
lstm.bias_ih_l0_reverse: (512,)
lstm.bias_hh_l0_reverse: (512,)


Total parameter count, worked out by hand (David): per direction `ih` 512×32 + `hh` 512×128 + two biases 512 each; × 2 directions. General form `2 × 4h(d + h + 2)`.

In [5]:
# 2 directions x (weight_ih 4h x d  +  weight_hh 4h x h)  +  4 bias vectors of 4h   (d = 32 input, h = 128 hidden, 4h = 512)
EXPECTED_BILSTM_PARAMS = 2 * (512 * 32) + 2 * (512 * 128) + 4 * 512   # = 165,888
assert sum(p.numel() for p in bilstm.parameters()) == EXPECTED_BILSTM_PARAMS
print("BiLSTM parameter count: passed")

BiLSTM parameter count: passed
